## 1. Initialize Project Environment
Import libraries for sequence handling, MSA via EBI REST API, and alignment analysis.

> **Note:** Initially planned to use Clustal Omega web interface manually, but automated via EBI's REST API instead for reproducibility.

In [1]:
from __future__ import annotations

import logging
import time
import urllib.parse
import urllib.request
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
from Bio import AlignIO, SeqIO
from Bio.SeqRecord import SeqRecord

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

## 2. Define Configuration Parameters
Centralize paths and MSA analysis options.

In [2]:
@dataclass
class MSAConfig:
    handle: str
    email: str = "student@example.com"
    fasta_path: Path = None
    export_dir: Path = Path("artifacts")
    conservation_threshold: float = 0.9
    trim_bp: Optional[int] = 1500  # Trim for Clustal Omega API
    ebi_api_url: str = "https://www.ebi.ac.uk/Tools/services/rest/clustalo"
    poll_interval: int = 5  # seconds between status checks

    def __post_init__(self):
        if self.fasta_path is None:
            self.fasta_path = Path(
                f"../../../data/work/{self.handle}/lab04/tp53_multi_sequences.fasta"
            )

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["fasta_path"] = str(info["fasta_path"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = MSAConfig(handle="AndreiCod")
CONFIG.describe()

{'handle': 'AndreiCod',
 'email': 'student@example.com',
 'fasta_path': '../../../data/work/AndreiCod/lab04/tp53_multi_sequences.fasta',
 'export_dir': 'artifacts',
 'conservation_threshold': 0.9,
 'trim_bp': 1500,
 'ebi_api_url': 'https://www.ebi.ac.uk/Tools/services/rest/clustalo',
 'poll_interval': 5}

In [3]:
def load_and_trim_sequences(cfg: MSAConfig) -> List[SeqRecord]:
    """Load sequences and optionally trim for Clustal Omega."""
    records = list(SeqIO.parse(cfg.fasta_path, "fasta"))

    if cfg.trim_bp:
        trimmed = []
        for rec in records:
            if len(rec.seq) > cfg.trim_bp:
                new_rec = rec[: cfg.trim_bp]
                new_rec.description = f"{rec.description} [trimmed to {cfg.trim_bp}bp]"
            else:
                new_rec = rec
            trimmed.append(new_rec)
        records = trimmed

    logging.info(
        "Loaded %d sequences (trimmed to %s bp)", len(records), cfg.trim_bp or "full"
    )
    return records


sequences = load_and_trim_sequences(CONFIG)
for rec in sequences:
    print(f"  {rec.id}: {len(rec.seq)} bp")

[INFO] Loaded 10 sequences (trimmed to 1500 bp)


  NM_000546.6: 1500 bp
  NM_011640.3: 1500 bp
  NM_131327.2: 1500 bp
  XM_006719566.3: 1500 bp
  NM_001317019.1: 1500 bp
  NM_001085860.1: 1500 bp
  XM_005194938.2: 1500 bp
  NM_001006919.1: 1500 bp
  NM_001089263.1: 1500 bp
  XM_031279688.1: 808 bp


## 3. Submit to Clustal Omega REST API
Use EBI's programmatic REST API instead of manual web upload.

In [4]:
def submit_clustalo_job(sequences: List[SeqRecord], cfg: MSAConfig) -> str:
    """Submit sequences to EBI Clustal Omega REST API and return job ID."""
    # Convert sequences to FASTA string
    from io import StringIO

    fasta_io = StringIO()
    SeqIO.write(sequences, fasta_io, "fasta")
    fasta_string = fasta_io.getvalue()

    # Prepare submission data
    params = {
        "email": cfg.email,
        "sequence": fasta_string,
        "outfmt": "clustal_num",
    }

    data = urllib.parse.urlencode(params).encode("utf-8")
    url = f"{cfg.ebi_api_url}/run"

    logging.info("Submitting %d sequences to Clustal Omega API...", len(sequences))
    req = urllib.request.Request(url, data=data, method="POST")
    with urllib.request.urlopen(req, timeout=60) as response:
        job_id = response.read().decode("utf-8").strip()

    logging.info("Job submitted: %s", job_id)
    return job_id


def poll_job_status(job_id: str, cfg: MSAConfig, max_wait: int = 300) -> str:
    """Poll EBI API until job completes. Returns final status."""
    url = f"{cfg.ebi_api_url}/status/{job_id}"
    elapsed = 0

    while elapsed < max_wait:
        with urllib.request.urlopen(url, timeout=30) as response:
            status = response.read().decode("utf-8").strip()

        if status == "FINISHED":
            logging.info("Job %s finished successfully", job_id)
            return status
        elif status in ("FAILURE", "ERROR"):
            raise RuntimeError(f"Clustal Omega job failed: {status}")

        logging.info("Job status: %s (waiting %ds...)", status, cfg.poll_interval)
        time.sleep(cfg.poll_interval)
        elapsed += cfg.poll_interval

    raise TimeoutError(f"Job {job_id} did not complete within {max_wait}s")


def fetch_alignment_result(job_id: str, cfg: MSAConfig) -> str:
    """Fetch alignment result from completed job."""
    url = f"{cfg.ebi_api_url}/result/{job_id}/aln-clustal_num"
    with urllib.request.urlopen(url, timeout=60) as response:
        result = response.read().decode("utf-8")
    return result


# Submit job and wait for result
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

job_id = submit_clustalo_job(sequences, CONFIG)
poll_job_status(job_id, CONFIG)
alignment_text = fetch_alignment_result(job_id, CONFIG)

# Save alignment
msa_result_path = EXPORT_DIR / "task3_msa_result.clustal"
with open(msa_result_path, "w") as f:
    f.write(alignment_text)

print(f"[OK] MSA result saved to: {msa_result_path.resolve()}")
print(f"\nFirst 500 characters of alignment:\n{alignment_text[:500]}")

[INFO] Submitting 10 sequences to Clustal Omega API...
[INFO] Job submitted: clustalo-R20251227-163132-0120-47250041-p1m
[INFO] Job status: QUEUED (waiting 5s...)
[INFO] Job status: RUNNING (waiting 5s...)
[INFO] Job clustalo-R20251227-163132-0120-47250041-p1m finished successfully


[OK] MSA result saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task3_msa_result.clustal

First 500 characters of alignment:
CLUSTAL O(1.2.4) multiple sequence alignment


NM_000546.6         --------------------------------------CTCAAAAGTCTAGAGCCACCGT	22
NM_011640.3         TTTCCCCTCCCACGTGCTCACCCTGGCTAAAGTTCTGTAGCTTCAGTTCATTGGGACCAT	60
NM_131327.2         ----------------------------------CTGTAACTAGG---------------	11
XM_006719566.3      ---------------------------------GCTACGTCATTACCAGGCACGCGCAGG	27
NM_001317019.1      ----------------------------------CTGTTTTTGGAAGGAAGAAAGATGGA	26
NM_001085860.1      -------------


## 4. Analyze MSA Conservation
Load the alignment result and identify conserved regions.

In [5]:
# Load alignment from saved file
alignment = AlignIO.read(msa_result_path, "clustal")
logging.info(
    "Loaded MSA: %d sequences × %d positions",
    len(alignment),
    alignment.get_alignment_length(),
)
print(
    f"Alignment dimensions: {len(alignment)} seqs × {alignment.get_alignment_length()} bp"
)
print(f"\nFirst 80 columns:\n{alignment[:, :80]}")

[INFO] Loaded MSA: 10 sequences × 1837 positions


Alignment dimensions: 10 seqs × 1837 bp

First 80 columns:
Alignment with 10 rows and 80 columns
--------------------------------------CTCAAA...GC- NM_000546.6
TTTCCCCTCCCACGTGCTCACCCTGGCTAAAGTTCTGTAGCTTC...AC- NM_011640.3
----------------------------------CTGTAACTAG...CC- NM_131327.2
---------------------------------GCTACGTCATT...GG- XM_006719566.3
----------------------------------CTGTTTTTGG...GC- NM_001317019.1
-------------------------ACTGGTGCCCTGA-GCAAG...GA- NM_001085860.1
---------------------CATGGCTGAGGGCCGGCGGCGGG...GC- XM_005194938.2
-------------------------------------CTCCCTG...GGT NM_001006919.1
---------GAA--AGGGAGGAAGTGCTGAATTGCTTCGCTTGG...GCC NM_001089263.1
-----GC-------------------------------------...AT- XM_031279688.1


In [6]:
def compute_conservation(alignment, threshold: float = 0.9) -> pd.DataFrame:
    """Calculate conservation score for each position."""
    positions = []
    aln_len = alignment.get_alignment_length()

    for pos in range(aln_len):
        column = alignment[:, pos]
        counts = pd.Series(list(column)).value_counts()
        top_base = counts.index[0]
        score = counts.iloc[0] / counts.sum()
        positions.append(
            {
                "position": pos,
                "top_base": top_base,
                "score": score,
                "conserved": score >= threshold,
            }
        )

    return pd.DataFrame(positions)


if "alignment" in dir():
    conservation_df = compute_conservation(alignment, CONFIG.conservation_threshold)
    print(f"Total positions: {len(conservation_df)}")
    print(
        f"Conserved positions (≥{CONFIG.conservation_threshold}): {conservation_df['conserved'].sum()}"
    )
    conservation_df.head(20)

Total positions: 1837
Conserved positions (≥0.9): 69


In [7]:
def find_conserved_blocks(
    conservation_df: pd.DataFrame, min_length: int = 10
) -> List[Dict]:
    """Find consecutive runs of conserved positions."""
    blocks = []
    in_block = False
    block_start = 0

    for i, row in conservation_df.iterrows():
        if row["conserved"] and not in_block:
            in_block = True
            block_start = row["position"]
        elif not row["conserved"] and in_block:
            in_block = False
            block_len = row["position"] - block_start
            if block_len >= min_length:
                blocks.append(
                    {
                        "start": block_start,
                        "end": row["position"] - 1,
                        "length": block_len,
                    }
                )

    # Handle last block
    if in_block:
        block_len = len(conservation_df) - block_start
        if block_len >= min_length:
            blocks.append(
                {
                    "start": block_start,
                    "end": len(conservation_df) - 1,
                    "length": block_len,
                }
            )

    return blocks


if "conservation_df" in dir():
    conserved_blocks = find_conserved_blocks(conservation_df)
    print(f"Found {len(conserved_blocks)} conserved blocks (≥10 bp):")
    for block in conserved_blocks[:10]:
        print(f"  Position {block['start']}-{block['end']} ({block['length']} bp)")

Found 0 conserved blocks (≥10 bp):


## 5. Summarize Conservation Statistics
Calculate summary metrics for notes.

In [8]:
def summarize_conservation(
    conservation_df: pd.DataFrame, conserved_blocks: List[Dict], threshold: float
) -> Dict:
    """Generate compact summary statistics for notes.md."""
    total_pos = len(conservation_df)
    conserved_count = conservation_df["conserved"].sum()
    conserved_pct = 100 * conserved_count / total_pos
    mean_score = conservation_df["score"].mean()

    summary = {
        "total_positions": total_pos,
        "conserved_positions": int(conserved_count),
        "conserved_percent": round(conserved_pct, 1),
        "mean_conservation_score": round(mean_score, 3),
        "num_conserved_blocks": len(conserved_blocks),
        "threshold_used": threshold,
    }

    # Top 3 longest conserved blocks
    top_blocks = sorted(conserved_blocks, key=lambda x: x["length"], reverse=True)[:3]
    summary["top_blocks"] = [
        f"{b['start']}-{b['end']} ({b['length']}bp)" for b in top_blocks
    ]

    return summary


# Generate summary
summary_stats = summarize_conservation(
    conservation_df, conserved_blocks, CONFIG.conservation_threshold
)

print("Conservation Summary:")
for k, v in summary_stats.items():
    print(f"  {k}: {v}")

Conservation Summary:
  total_positions: 1837
  conserved_positions: 69
  conserved_percent: 3.8
  mean_conservation_score: 0.507
  num_conserved_blocks: 0
  threshold_used: 0.9
  top_blocks: []


## 6. Export Results
Save conservation data to artifacts.

In [9]:
# Save conservation scores CSV
cons_path = EXPORT_DIR / "task3_conservation_scores.csv"
conservation_df.to_csv(cons_path, index=False)
print(f"[OK] Conservation scores saved to: {cons_path.resolve()}")

# Save summary as JSON for reproducibility
import json

summary_path = EXPORT_DIR / "task3_conservation_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary_stats, f, indent=2)
print(f"[OK] Conservation summary saved to: {summary_path.resolve()}")

print(f"\nArtifacts saved to {EXPORT_DIR.resolve()}")

[OK] Conservation scores saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task3_conservation_scores.csv
[OK] Conservation summary saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts/task3_conservation_summary.json

Artifacts saved to /home/rbals/git/daha-bdhb/BDHB-lab/labs/04_phylogenetics/assignments/artifacts
